# 15 — ONE-TIME Sealed Holdout Evaluation

**WARNING:** This notebook opens the internal sealed holdout for the first and only model-selection-independent evaluation.

Locked pipeline is loaded from:
`cv_outputs/final_ensemble_lock/locked_pipeline.json`

Rules enforced here:
- NO threshold scan
- NO weight optimization
- NO hyperparameter tuning
- NO candidate comparison
- NO TTA
- NO training
- exactly the locked preprocessing / model weights / threshold
- holdout predictions are generated once and saved

Inference structure:
1. For each seed, average predictions from its 5 fold checkpoints.
2. Combine seed-level probabilities using locked weights.
3. Apply locked threshold.
4. Report the competition metric and confusion matrix.

After this notebook runs, holdout is considered **USED** and must not be used for further tuning.


In [ ]:
print('>>> CELL 1 START: load locked pipeline', flush=True)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

from pathlib import Path
import json,time,gc
import numpy as np
import pandas as pd
from PIL import Image,ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,models
from sklearn.metrics import confusion_matrix,f1_score,roc_auc_score

ROOT=Path(r'D:\medicine')
MANIFEST=ROOT/'split_outputs'/'split_manifest.csv'
LOCK_PATH=ROOT/'cv_outputs'/'final_ensemble_lock'/'locked_pipeline.json'
OUT=ROOT/'holdout_outputs'/'locked_pipeline_v1'
OUT.mkdir(parents=True,exist_ok=True)

with open(LOCK_PATH,'r',encoding='utf-8') as f:
    lock=json.load(f)

assert lock['status']=='LOCKED_BEFORE_HOLDOUT'
assert lock['holdout_used'] is False
assert lock['architecture']=='DenseNet121'
assert lock['image_size']==224
assert lock['preprocessing']=='force_grayscale_then_rgb'
assert lock['tta'] is False
assert len(lock['models'])==2

IMAGE_SIZE=int(lock['image_size'])
THRESHOLD=float(lock['threshold'])
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
BATCH_SIZE=16

print('Device:',DEVICE,flush=True)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('LOCKED weights:',[(m['name'],m['weight']) for m in lock['models']],flush=True)
print('LOCKED threshold:',THRESHOLD,flush=True)


In [ ]:
print('>>> CELL 2 START: load sealed holdout ONLY', flush=True)
manifest=pd.read_csv(MANIFEST)
dev=manifest[manifest.split=='dev'].copy().reset_index(drop=True)
holdout=manifest[manifest.split=='holdout'].copy().reset_index(drop=True)

assert len(dev)==4362
assert len(holdout)==770
assert set(dev.sha256).isdisjoint(set(holdout.sha256))
assert holdout.filename.nunique()==770

print('SEALED HOLDOUT OPENED:',len(holdout),flush=True)
print('Labels:',holdout.label.value_counts().to_dict(),flush=True)


In [ ]:
print('>>> CELL 3 START: preprocess holdout exactly as locked', flush=True)

class PadToSquare:
    def __call__(self,img):
        w,h=img.size
        s=max(w,h)
        l=(s-w)//2; r=s-w-l
        t=(s-h)//2; b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

pad=PadToSquare()
buf=[]
t0=time.time()

for i,row in holdout.iterrows():
    with Image.open(row.path) as im:
        im=im.convert('L').convert('RGB')
        im=pad(im)
        im=im.resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.BILINEAR)
        buf.append(np.asarray(im,dtype=np.uint8))

    if (i+1)%200==0 or i+1==len(holdout):
        print(f'  cached {i+1}/{len(holdout)}',flush=True)

HOLDOUT_CACHE=np.stack(buf)
del buf
print(f'Holdout cache: {HOLDOUT_CACHE.shape} | {time.time()-t0:.1f}s',flush=True)


In [ ]:
print('>>> CELL 4 START: inference helpers', flush=True)
MEAN=[0.485,0.456,0.406]
STD=[0.229,0.224,0.225]
L2I={'Normal':0,'Pneumonia':1}

val_tf=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])

class HoldoutDS(Dataset):
    def __init__(self,frame):
        self.df=frame.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        im=Image.fromarray(HOLDOUT_CACHE[i])
        return val_tf(im),L2I[row.label],row.filename

loader=DataLoader(
    HoldoutDS(holdout),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

def make_model():
    m=models.densenet121(weights=None)
    m.classifier=nn.Linear(m.classifier.in_features,2)
    return m.to(DEVICE)

@torch.no_grad()
def infer_checkpoint(checkpoint_path):
    ck=torch.load(checkpoint_path,map_location='cpu')
    model=make_model()
    model.load_state_dict(ck['state_dict'])
    model.eval()

    ys=[];ps=[];names=[]
    for x,y,n in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=model(x)
        p=torch.softmax(logits.float(),dim=1)[:,1]
        ys.extend(y.numpy())
        ps.extend(p.cpu().numpy())
        names.extend(list(n))

    del model,ck
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

    return np.asarray(ys),np.asarray(ps),names

def competition_metrics(y,p,t):
    y=np.asarray(y,int);p=np.asarray(p,float)
    pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y)
    se=tp/max(tp+fn,1)
    sp=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    auc=roc_auc_score(y,p)
    return {
        'P':float(P),
        'Accuracy':float(acc),
        'Sensitivity':float(se),
        'Specificity':float(sp),
        'F1':float(f1),
        'AUROC':float(auc),
        'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
        'threshold':float(t)
    }


In [ ]:
print('>>> CELL 5 START: ONE-TIME locked holdout inference', flush=True)

seed_outputs={}
y_ref=None
names_ref=None

for spec in lock['models']:
    name=spec['name']
    ckpt_dir=Path(spec['checkpoint_dir'])
    fold_probs=[]

    print('\nMODEL GROUP:',name,flush=True)

    for fold in range(5):
        ckpt=ckpt_dir/f'fold{fold}_best.pt'
        assert ckpt.exists(),f'Missing checkpoint: {ckpt}'
        print(f'  inference fold {fold}/4 ...',flush=True)

        y,p,names=infer_checkpoint(ckpt)

        if y_ref is None:
            y_ref=y
            names_ref=names
        else:
            assert np.array_equal(y_ref,y)
            assert names_ref==names

        fold_probs.append(p)

    seed_prob=np.mean(np.stack(fold_probs,axis=1),axis=1)
    seed_outputs[name]=seed_prob
    print('  finished',name,flush=True)

assert len(seed_outputs)==2

final_prob=np.zeros(len(y_ref),dtype=float)
for spec in lock['models']:
    final_prob += float(spec['weight'])*seed_outputs[spec['name']]

metrics=competition_metrics(y_ref,final_prob,THRESHOLD)
pred=(final_prob>=THRESHOLD).astype(int)

pred_df=pd.DataFrame({
    'filename':names_ref,
    'y_true':y_ref,
    'p_locked_ensemble':final_prob,
    'prediction':pred
})

for name,p in seed_outputs.items():
    pred_df[f'p_{name}_5fold_avg']=p

pred_df.to_csv(OUT/'holdout_predictions.csv',index=False)

result={
    'evaluation':'ONE_TIME_SEALED_HOLDOUT',
    'pipeline_lock_file':str(LOCK_PATH),
    'architecture':lock['architecture'],
    'image_size':IMAGE_SIZE,
    'preprocessing':lock['preprocessing'],
    'model_weights':[
        {'name':m['name'],'weight':float(m['weight'])}
        for m in lock['models']
    ],
    'threshold':THRESHOLD,
    'tta':False,
    'holdout_images':int(len(y_ref)),
    'metrics':metrics,
    'NO_RETUNING_AFTER_THIS':True,
    'holdout_used':True
}

with open(OUT/'holdout_evaluation.json','w',encoding='utf-8') as f:
    json.dump(result,f,indent=2)

print('\n'+'='*80,flush=True)
print('ONE-TIME SEALED HOLDOUT RESULT',flush=True)
print('='*80,flush=True)
print(json.dumps(result,indent=2),flush=True)


## STOP after evaluation

After running this notebook:

- Do **not** change ensemble weights based on holdout.
- Do **not** change threshold based on holdout.
- Do **not** select another model because of holdout performance.
- Do **not** run alternative holdout experiments.

Use the result only as an unbiased estimate of the locked pipeline's generalization.

Next step is final official-test inference / submission packaging using the already locked pipeline.
